# Stage B — focal loss alpha=0.75, on Google Colab (GPU)

Same config as `scripts/hpc/train_stage_b_alpha75.slurm` (HPC job `1802232`, still queued
behind other jobs) — run here instead so we don't have to wait on the cluster queue.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine).

**One-time setup this notebook needs from you** — upload these to your Google Drive once,
under a folder called `stage_b_data/` in "My Drive":
- `data/processed/stage_b/` (the whole folder — metadata.csv, images/, tile_labels.npy, stage_b_meta.json) — ~212 MB
- `weights/yolo11m.pt` — ~39 MB

So Drive ends up with:
```
My Drive/stage_b_data/data/processed/stage_b/...
My Drive/stage_b_data/weights/yolo11m.pt
```
(Both are git-ignored, which is why they aren't already in the repo clone below.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone -b feature/stage-a-mio-tcd https://github.com/pelemele1/Farhad_Vaseghi_MA.git /content/repo
%cd /content/repo

In [ ]:
# link the Drive-uploaded data/weights into the repo's expected paths
import os
DRIVE_ROOT = "/content/drive/MyDrive/stage_b_data"
assert os.path.isdir(f"{DRIVE_ROOT}/data/processed/stage_b"), (
    f"Expected {DRIVE_ROOT}/data/processed/stage_b on your Drive -- see the setup note above."
)
assert os.path.isfile(f"{DRIVE_ROOT}/weights/yolo11m.pt"), (
    f"Expected {DRIVE_ROOT}/weights/yolo11m.pt on your Drive -- see the setup note above."
)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("weights", exist_ok=True)
if not os.path.exists("data/processed/stage_b"):
    os.symlink(f"{DRIVE_ROOT}/data/processed/stage_b", "data/processed/stage_b")
if not os.path.exists("weights/yolo11m.pt"):
    os.symlink(f"{DRIVE_ROOT}/weights/yolo11m.pt", "weights/yolo11m.pt")
print("ok")

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- set Runtime > GPU")

## Train

Exact same hyperparameters as `train_stage_b_alpha75.slurm`: 40 epochs, batch size 16,
lr 1e-3, img-size 512, focal loss alpha=0.75, gamma=2.0.

In [ ]:
!python scripts/train_stage_b.py \
    --data data/processed/stage_b \
    --weights weights/yolo11m.pt \
    --epochs 40 \
    --batch-size 16 \
    --lr 1e-3 \
    --img-size 512 \
    --loss focal \
    --focal-alpha 0.75 \
    --focal-gamma 2.0 \
    --device cuda \
    --out checkpoints/stage_b_alpha75

## Evaluate (on Colab, quick check) and save the checkpoint back to Drive

In [ ]:
!python scripts/evaluate_stage_b.py \
    --checkpoint checkpoints/stage_b_alpha75/stage_b_head.pt \
    --data data/processed/stage_b --split test --device cuda

In [ ]:
# copy the trained checkpoint back to Drive so you can download it and pull it into the local repo
import shutil
os.makedirs(f"{DRIVE_ROOT}/checkpoints_out", exist_ok=True)
shutil.copy("checkpoints/stage_b_alpha75/stage_b_head.pt", f"{DRIVE_ROOT}/checkpoints_out/stage_b_head_alpha75.pt")
print(f"saved to {DRIVE_ROOT}/checkpoints_out/stage_b_head_alpha75.pt -- download it from Drive,\n"
      "then drop it in your local repo at checkpoints/stage_b_alpha75/stage_b_head.pt")